# **Raw Data Preparation for Pre-train the BERT Model**

#### **Import Libraries**

In [1]:
import torch.nn as nn
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.utils import get_tokenizer
from transformers import BertTokenizer
from torchtext.datasets import IMDB
import random
import torch

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.0
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
tokenizor = BertTokenizer.from_pretrained("bert-base-uncased")

In [3]:
text = "transformers are powerful models. I am playing cricket"
tokens = tokenizor.tokenize(text)
tokens

['transformers',
 'are',
 'powerful',
 'models',
 '.',
 'i',
 'am',
 'playing',
 'cricket']

In [4]:
#Dataset unpack
train_iter, val_iter = IMDB()
train_iter

ShardingFilterIterDataPipe

In [5]:
def yield_tokens(data_iter):
    for label, text in data_iter:
        yield tokenizor.tokenize(text)

In [6]:
PAD_IDX, CLS_IDX, SEP_IDX, MASK_IDX, UNK_IDX = 0,1,2,3,4
special_tokens = ['PAD_IDX','CLS_IDX', 'SEP_IDX', 'MASK_IDX', 'UNK_IDX']


vocab = build_vocab_from_iterator(yield_tokens(train_iter),
                                  specials= special_tokens,
                                  special_first=True, max_tokens=1000)
vocab.set_default_index(UNK_IDX)

vocab_size = len(vocab)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (718 > 512). Running this sequence through the model will result in indexing errors


In [7]:
vocab(tokens)
vocab.get_itos()[53]


'there'

### **Text Masking** 

In [8]:
def bernoulli_true_false (percentage):
    number:bool = random.random() < percentage
    return number
    

In [9]:
def index_to_text(index):
    return vocab.get_itos()[index]

In [10]:
def text_masking(token):
    mask = bernoulli_true_false(0.2)
    
    token_ = ""
    mask_label = ""
    
    if not mask:
        token_ = token
        mask_label = '[PAD]'# if mask false then return as it is and the label also
    
    random_opp = bernoulli_true_false(0.5)
    random_switch = bernoulli_true_false(0.5)
    
    # case 1: if mask is true, random_opp is true and random_sqitch is true
    if mask and random_opp and random_switch:
        token_ = '[MASK]'
        mask_label =  index_to_text(torch.randint(0, vocab_size,(1,)))
        
    #case 2: if mask is true, random_opp is true and random_switch is false
    elif mask and random_opp and not random_switch:
        token_ = token
        mask_label = token
        
    #case 3: if mask is true and random_opp is false
    elif mask and not random_opp:
        token_ = '[MASK]'
        mask_label = token
    
    return token_, mask_label
    
    

In [11]:
text_masking('playing')

('[MASK]', 'must')

### **MLM Preparation**

In [12]:
def mlm_preparation (tokens, include_raw_tokens = False):
    
    bert_input = []
    bert_label = []
    raw_tokens = []
    bert_current_input = []
    bert_current_label = []
    current_raw_tokens = []
    
    for token in tokens:
        
        mask_token, mask_label = text_masking(token)
        
        bert_current_input.append(mask_token)
        bert_current_label.append(mask_label)
        
        if include_raw_tokens:
            current_raw_tokens.append(token)
            
        # check whether the tokens is sentence delimeter
        if token in ['.','?','!']:
            
            if len(bert_current_input) >  2:
                bert_input.append(bert_current_input)
                bert_label.append(bert_current_label)
                
                if include_raw_tokens:
                    raw_tokens.append(current_raw_tokens)
                
                # reset the list for next sentence
                bert_current_input = []
                bert_current_label = []
                current_raw_tokens = []
            
            # length is not enough to create a sentence
            else:
                bert_current_input = []
                bert_current_label = []
                current_raw_tokens = []               
            
    if bert_current_input:
        bert_input.append(bert_current_input)
        bert_label.append(bert_current_label)
            
        if include_raw_tokens:
                raw_tokens.append(current_raw_tokens)       
        
    return (bert_input, bert_label, raw_tokens) if include_raw_tokens else (bert_input,bert_label)
    
    

In [13]:
bert_input, bert_label = mlm_preparation(tokens) 
bert_input

[['transformers', 'are', 'powerful', 'models', '.'],
 ['i', 'am', 'playing', 'cricket']]

In [14]:
bert_label

[['[PAD]', '[PAD]', '[PAD]', '[PAD]', '.'],
 ['[PAD]', '[PAD]', '[PAD]', '[PAD]']]

### **Function for Next Sentence Prediction**

In [15]:
# in here we are looking to evaluate and comparison between bert_input and bert_label 
def nsp_preparation (input_sentence, input_mask_label):
    
    if len(input_sentence) < 2:
        raise ValueError("input sentence size must be larger than 2")
    
    if (len(input_sentence) != len(input_mask_label)):
        raise ValueError ("input sentence and input mask label must have the same length")

    bert_input, bert_label, is_next = [], [], []
    
    available_indices = list(range(len(input_sentence)))
    
    while len(available_indices) >= 2:
        
        if random.random() < 0.5:
            index = random.choice(available_indices[:-1])
           
            bert_input.append([['[CLS]'] + input_sentence[index]+['[SEP]'],
                            input_sentence[index+1]+['[SEP]']])
            
            bert_label.append([['[PAD]'] + input_mask_label[index] +['[PAD]'],
                               input_mask_label[index +1]+ ['[PAD]']])
            
            is_next.append(1) # 1 label indicate two sentences are consecutive
            
            available_indices.remove(index)
            if (index+1) in available_indices:
                available_indices.remove(index+1)
        
        else:
            
            indices = random.sample(available_indices, 2)
            
            bert_input.append([['[CLS]'] + input_sentence[indices[0]] + ['[SEP]'],
                               input_sentence[indices[1]]+ ['[SEP]']])
            
            bert_label.append([['[PAD]'] + input_mask_label[indices[0]]+ ['[PAD]'],
                               input_mask_label[indices[1]]+['[PAD]']])
            
            is_next.append(0)
            
            available_indices.remove(indices[0])
            available_indices.remove(indices[1])
    
    return bert_input, bert_label, is_next 

In [20]:
bert_input_nsp, bert_label_nsp, is_next = nsp_preparation(bert_input,bert_label)

In [21]:
bert_input_nsp

[[['[CLS]', 'transformers', 'are', 'powerful', 'models', '.', '[SEP]'],
  ['i', 'am', 'playing', 'cricket', '[SEP]']]]

In [22]:
bert_label_nsp

[[['[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '.', '[PAD]'],
  ['[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']]]

In [23]:
is_next

[0]

### **Build Final Bert Input and Output** 

* So in there we need to convert our tokenized prepared inputs to a tensor with numerical values. 

* Ultimately these values are going to feed the input as the encoder.

In [ ]:
def final_bert_input(bert_input, bert_label, is_next, to_tensor = True):
    
    def zero_pad_list_pair(pair:list, pad = '[PAD]'):
        #extract the maximum length between the pair of lists
        max_len = max(len(pair[0]), len(pair[1])) 
        
        pair[0].extend([pad] * (max_len - len(pair[0])))
        pair[1].extend([pad] * (max_len- len(pair[1])))
        
        return pair[0], pair[1]

    #flattern the pair lists
    flattern = lambda list_items: [item for sublist in list_items for item in sublist]
    #convert text tokens into indices
    tokens_to_index = lambda tokens: [vocab[token] for token in tokens]
    
    